In [1]:
"""1-3. 색인 대조 -- json 만 읽으므로 개발 컨테이너의 아무 python 에서나 돈다"""
import json

# pod 머지가 남긴 색인 (회수물) 과 재머지가 만든 색인의 경로
REC = (
    "/workspace/study/physical-ai-study/Studies/Phase 4.5/week3/outputs/recovered/runs/"
    "openvla-7b+maniskill_pickcube_only+b16+lr-0.0005+lora-r32+dropout-0.0--image_aug"
)
DST = "/workspace/models/openvla-maniskill-ft"

old = json.load(open(f"{REC}/model.safetensors.index.json"))   # pod 머지가 남긴 색인
new = json.load(open(f"{DST}/model.safetensors.index.json"))   # 재머지가 만든 색인

# 총 바이트: dtype 이나 텐서 크기가 하나라도 다르면 어긋난다
print("total_size:", old["metadata"]["total_size"], new["metadata"]["total_size"])
print("total_size 일치:", old["metadata"]["total_size"] == new["metadata"]["total_size"])

# 텐서 이름 집합: 어댑터가 덜 합쳐졌으면 이름이 남거나 빠진다
old_keys, new_keys = set(old["weight_map"]), set(new["weight_map"])
print("텐서 이름 집합 일치:", old_keys == new_keys)
print("한쪽에만 있는 이름 수:", len(old_keys ^ new_keys))

total_size: 15082474368 15082474368
total_size 일치: True
텐서 이름 집합 일치: True
한쪽에만 있는 이름 수: 0


In [2]:
import torch
from transformers import AutoModelForVision2Seq
from transformers import AutoProcessor
from transformers import BitsAndBytesConfig

MODEL_PATH = "/workspace/models/openvla-maniskill-ft"
BASELINE_GB = 4.38

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.reset_peak_memory_stats()
before_gb = torch.cuda.memory_allocated()
print(f"적재 전: {before_gb:.2f} GB")

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH,
    attn_implementation="eager",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=bnb_config,
)
after_gb = torch.cuda.memory_allocated() / 1e9
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"적재 후: {after_gb:.2f} GB (피크 {peak_gb:.2f} GB)")

diff = after_gb - BASELINE_GB
print(f"baseline({BASELINE_GB} GB) 대비 차이: {diff:+.2f} GB")
if abs(diff) < 0.5:
    print("판정: 근사 일치 -- 양자화 적용됨")
else:
    print("판정: 벗어남 -- 양자화 설정 또는 적재 범위 확인 필요")

/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


적재 전: 0.00 GB


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  

적재 후: 4.38 GB (피크 4.38 GB)
baseline(4.38 GB) 대비 차이: +0.00 GB
판정: 근사 일치 -- 양자화 적용됨


In [7]:
import json
import numpy as np
import torch
from PIL import Image

STATS_PATH = "/workspace/models/openvla-maniskill-ft/dataset_statistics.json"
DATASET_KEY = "maniskill_pickcube"
INSTRUCTION = "pick up the cube"

print(f"현재 norm_stats 키: ", list(vla.norm_stats.keys()))

with open(STATS_PATH) as f:
    stats = json.load(f)
print("통계 파일 키: ", list(stats.keys()))
vla.norm_stats[DATASET_KEY] = stats[DATASET_KEY]
print("주입 후 키: ", list(vla.norm_stats.keys()))

image = Image.fromarray(
    (np.random.RandomState(0).rand(224, 224, 3) * 255).astype(np.uint8)
)
prompt = f"In: What action should the robot take to {INSTRUCTION}?\nOut:"
inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)

for key in ["bridge_orig", DATASET_KEY]:
    with torch.no_grad():
        action = vla.predict_action(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            unnorm_key=key,
            do_sample=False,
        )
    print(f"unnorm_key={key}")
    print("action: ", np.round(action, 4))
    print("위치 3차원 크기: ", np.round(np.abs(action[:3]), 4))

action_stats = stats[DATASET_KEY]["action"]
for name, value in action_stats.items():
    value = np.asarray(value)    
    if value.dtype == bool:
        print(f"{name}: {value}")
    else:
        print(f"{name}: {np.round(value, 4)}")

현재 norm_stats 키:  ['austin_buds_dataset_converted_externally_to_rlds', 'austin_sailor_dataset_converted_externally_to_rlds', 'austin_sirius_dataset_converted_externally_to_rlds', 'bc_z', 'berkeley_autolab_ur5', 'berkeley_cable_routing', 'berkeley_fanuc_manipulation', 'bridge_orig', 'cmu_stretch', 'dlr_edan_shared_control_converted_externally_to_rlds', 'dobbe', 'fmb_dataset', 'fractal20220817_data', 'furniture_bench_dataset_converted_externally_to_rlds', 'iamlab_cmu_pickup_insert_converted_externally_to_rlds', 'jaco_play', 'kuka', 'nyu_franka_play_dataset_converted_externally_to_rlds', 'roboturk', 'stanford_hydra_dataset_converted_externally_to_rlds', 'taco_play', 'toto', 'ucsd_kitchen_dataset_converted_externally_to_rlds', 'utaustin_mutex', 'viola', 'maniskill_pickcube']
통계 파일 키:  ['maniskill_pickcube']
주입 후 키:  ['austin_buds_dataset_converted_externally_to_rlds', 'austin_sailor_dataset_converted_externally_to_rlds', 'austin_sirius_dataset_converted_externally_to_rlds', 'bc_z', 'berkel

In [8]:
import accelerate
import bitsandbytes
import numpy as np
import timm
import tokenizers
import torch
import transformers

print("추론 환경")

for module in [torch, transformers, tokenizers, timm, accelerate, bitsandbytes]:
    print(f"{module.__name__}: {module.__version__}")

outputs = []
for trial in range(3):
    with torch.no_grad():
        action = vla.predict_action(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            unnorm_key=DATASET_KEY,
            do_sample=False,
        )
    outputs.append(np.asarray(action))
    print(f"trial{trial}: {np.round(action, 5)}")

max_diff = max(np.abs(outputs[0] - other).max() for other in outputs[1:])
print(f"최대 편차: {max_diff:.2e}")
print("판정: ", "결정적" if max_diff < 1e-6 else "비결정 -- do_sample / dropout 설정 확인")

추론 환경
torch: 2.12.0+cu130
transformers: 4.40.1
tokenizers: 0.19.1
timm: 0.9.16
accelerate: 1.0.1
bitsandbytes: 0.49.2


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trial0: [-5.500e-04  1.187e-02 -1.184e-02 -5.000e-05  2.300e-04  7.800e-04
  0.000e+00]
trial1: [-5.500e-04  1.187e-02 -1.184e-02 -5.000e-05  2.300e-04  7.800e-04
  0.000e+00]
trial2: [-5.500e-04  1.187e-02 -1.184e-02 -5.000e-05  2.300e-04  7.800e-04
  0.000e+00]
최대 편차: 0.00e+00
판정:  결정적


In [10]:
import numpy as np
import torch
import gymnasium as gym
import mani_skill.envs
from PIL import Image

ENV_ID = "PickCube-v1"
MAX_EPISODE_STEPS = 200
ACTION_REPEAT = 4
POLICY_STEPS = MAX_EPISODE_STEPS // ACTION_REPEAT
SMOKE_SEED = 500
INSTRUCTION = "pick up the cube"

MODEL_PATH = "/workspace/models/openvla-maniskill-ft"
UNNORM_KEY = "maniskill_pickcube"

env = gym.make(
    ENV_ID,
    obs_mode="rgb",
    control_mode="pd_ee_delta_pose",
    render_mode="rgb_array",
    sensor_configs=dict(width=224, height=224),
    max_episode_steps=MAX_EPISODE_STEPS,
)
obs, info = env.reset(seed=SMOKE_SEED)
prompt = f"In: What action should the robot take to {INSTRUCTION}?\nOut:"

done = False
for policy_step in range(POLICY_STEPS):
    frame = obs["sensor_data"]["base_camera"]["rgb"].cpu().numpy()
    if frame.ndim == 4:
        frame = frame[0]
    image = Image.fromarray(frame.astype(np.uint8)).resize((224, 224))
    model_inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)
    with torch.no_grad():
        raw_action = vla.predict_action(
            input_ids=model_inputs["input_ids"],
            pixel_values=model_inputs["pixel_values"],
            unnorm_key=UNNORM_KEY,
            do_sample=False,
        )
    action = raw_action
    for _ in range(ACTION_REPEAT):
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated or bool(info["success"].item()):
            done = True
            break
    if done:
        break

env.close()
print(f"루프 완주: 정책 결정 {policy_step + 1}회. 예외 없음")
print("성공/실패는 판정하지 않는다 -- week5의 N회 측정에서 다룬다")

/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


루프 완주: 정책 결정 50회. 예외 없음
성공/실패는 판정하지 않는다 -- week5의 N회 측정에서 다룬다
